# Big Data Essentils Group Assignment

## Group 11

### Task 1 HDFS + Spark Data Ingestion

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_date, date_format, hour, month, dayofweek

spark = SparkSession.builder \
    .appName("RRSIS") \
    .master("local[*]") \
    .getOrCreate()

df = spark.read.csv(
    "hdfs://localhost:9000/rrsis/data/Road Accident Data.csv",
    header=True,
    inferSchema=True
)
df = df.withColumn("Time", col("Time").cast("string"))

df_clean = df.withColumn("Accident_Date_Parsed", to_date(col("Accident Date"), "M/d/yyyy"))
df_clean = df_clean.withColumn("Time_of_Day", date_format(col("Time"), "HH:mm:ss"))
df_clean = df_clean.dropDuplicates()
df_clean = df_clean.filter(col("Time").isNotNull())

for c in ["Road_Surface_Conditions", "Road_Type", "Weather_Conditions"]:
    df_clean = df_clean.withColumn(
        c,
        when((col(c).isNull()) | (col(c) == ""), "Unknown").otherwise(col(c))
    )

df_accidents = df_clean.dropDuplicates(["Accident_Index"])
print("Accident-level rows:", df_accidents.count())

C:\Users\Christian\anaconda3\envs\ml_env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Accident-level rows: 197644


## Create the time-period buckets
### Extract hour, weekday, month from the cleaned columns

In [2]:
df_time = df_accidents.withColumn("Hour", hour(col("Time_of_Day")))
df_time = df_time.withColumn("Month_Num", month(col("Accident_Date_Parsed")))
df_time = df_time.withColumn(
    "Weekend_Flag",
    when(col("Day_of_Week").isin("Saturday", "Sunday"), "Weekend").otherwise("Weekday")
)
df_time.select("Accident_Date_Parsed", "Time_of_Day", "Hour", "Day_of_Week", "Month", "Weekend_Flag").show(10)

+--------------------+-----------+----+-----------+-----+------------+
|Accident_Date_Parsed|Time_of_Day|Hour|Day_of_Week|Month|Weekend_Flag|
+--------------------+-----------+----+-----------+-----+------------+
|          2022-01-05|   12:15:00|  12|    Tuesday|  Jan|     Weekday|
|          2021-03-18|   07:38:00|   7|  Wednesday|  Mar|     Weekday|
|          2021-03-19|   19:13:00|  19|   Thursday|  Mar|     Weekday|
|          2021-04-13|   16:30:00|  16|     Monday|  Apr|     Weekday|
|          2021-04-03|   23:30:00|  23|     Friday|  Apr|     Weekday|
|          2021-04-12|   09:04:00|   9|     Sunday|  Apr|     Weekend|
|          2022-01-13|   17:20:00|  17|  Wednesday|  Jan|     Weekday|
|          2021-04-20|   11:36:00|  11|     Monday|  Apr|     Weekday|
|          2021-01-08|   05:00:00|   5|   Thursday|  Jan|     Weekday|
|          2022-05-23|   11:40:00|  11|     Sunday|  May|     Weekend|
+--------------------+-----------+----+-----------+-----+------------+
only s

In [3]:
df_time = df_time.withColumn(
    "Time_Period",
    when((col("Hour") >= 0) & (col("Hour") < 5), "Late Night")
    .when((col("Hour") >= 5) & (col("Hour") < 12), "Morning")
    .when((col("Hour") >= 12) & (col("Hour") < 17), "Afternoon")
    .when((col("Hour") >= 17) & (col("Hour") < 21), "Evening")
    .otherwise("Night")
)
df_time.groupBy("Time_Period").count().orderBy(col("count").desc()).show()

+-----------+-----+
|Time_Period|count|
+-----------+-----+
|  Afternoon|66856|
|    Morning|56379|
|    Evening|48896|
|      Night|15799|
| Late Night| 9714|
+-----------+-----+



### accidents by hour

In [4]:
df_time.groupBy("Hour").count().orderBy(col("count").desc()).show(24)

+----+-----+
|Hour|count|
+----+-----+
|  17|17134|
|  16|15682|
|  15|15403|
|   8|14452|
|  18|13824|
|  13|12138|
|  14|12089|
|  12|11544|
|  19|10439|
|  11|10135|
|   9| 9998|
|  10| 8746|
|   7| 7995|
|  20| 7499|
|  21| 6215|
|  22| 5372|
|  23| 4212|
|   6| 3394|
|   0| 3058|
|   1| 2305|
|   2| 1679|
|   5| 1659|
|   3| 1522|
|   4| 1150|
+----+-----+



### accidents by day of week

In [5]:
df_time.groupBy("Day_of_Week").count().orderBy(col("count").desc()).show()

+-----------+-----+
|Day_of_Week|count|
+-----------+-----+
|     Friday|32376|
|    Tuesday|29853|
|  Wednesday|29850|
|   Thursday|29389|
|     Monday|28154|
|   Saturday|26544|
|     Sunday|21478|
+-----------+-----+



### accidents by month

In [6]:
df_time.groupBy("Month").count().orderBy(col("count").desc()).show(12)

+-----+-----+
|Month|count|
+-----+-----+
|  Nov|18650|
|  Oct|18310|
|  Jul|17356|
|  Sep|17129|
|  Jun|17122|
|  May|16838|
|  Mar|16567|
|  Aug|16238|
|  Apr|15496|
|  Jan|14962|
|  Dec|14881|
|  Feb|14095|
+-----+-----+



### weekday vs weekend

In [7]:
df_time.groupBy("Weekend_Flag").count().show()

+------------+------+
|Weekend_Flag| count|
+------------+------+
|     Weekday|149622|
|     Weekend| 48022|
+------------+------+



### the *"five highest-risk time periods"*
Combine more than one dimension

In [8]:
risk_periods = df_time.groupBy("Time_Period", "Weekend_Flag").count().orderBy(col("count").desc())
risk_periods.show(5)

+-----------+------------+-----+
|Time_Period|Weekend_Flag|count|
+-----------+------------+-----+
|  Afternoon|     Weekday|49971|
|    Morning|     Weekday|45953|
|    Evening|     Weekday|38147|
|  Afternoon|     Weekend|16885|
|      Night|     Weekday|11226|
+-----------+------------+-----+
only showing top 5 rows


### Assign severity weights

In [9]:
from pyspark.sql.functions import sum as spark_sum, count as spark_count

df_severity = df_time.withColumn(
    "Severity_Weight",
    when(col("Accident_Severity") == "Slight", 1)
    .when(col("Accident_Severity") == "Serious", 3)
    .when(col("Accident_Severity") == "Fatal", 5)
    .otherwise(0)
)

df_severity.groupBy("Accident_Severity", "Severity_Weight").count().show()

+-----------------+---------------+------+
|Accident_Severity|Severity_Weight| count|
+-----------------+---------------+------+
|          Serious|              3| 25685|
|            Fatal|              5|  2486|
|           Slight|              1|169473|
+-----------------+---------------+------+



### severity score by location (Local_Authority_(District))

In [10]:
severity_by_location = df_severity.groupBy("Local_Authority_(District)") \
    .agg(
        spark_count("Accident_Index").alias("Accident_Count"),
        spark_sum("Severity_Weight").alias("Severity_Score")
    ) \
    .orderBy(col("Severity_Score").desc())

severity_by_location.show(15, truncate=False)

+--------------------------+--------------+--------------+
|Local_Authority_(District)|Accident_Count|Severity_Score|
+--------------------------+--------------+--------------+
|Birmingham                |6165          |7805          |
|Westminster               |2811          |4341          |
|Manchester                |3132          |3854          |
|Sheffield                 |2750          |3464          |
|Liverpool                 |2611          |3445          |
|Cornwall                  |2606          |3288          |
|County Durham             |2228          |2904          |
|Lambeth                   |2250          |2884          |
|Barnet                    |2302          |2830          |
|Southwark                 |1977          |2579          |
|Camden                    |1671          |2529          |
|Doncaster                 |1945          |2509          |
|Nottingham                |1931          |2499          |
|Wiltshire                 |1590          |2390         

### severity score by road type

In [11]:
severity_by_roadtype = df_severity.groupBy("Road_Type") \
    .agg(
        spark_count("Accident_Index").alias("Accident_Count"),
        spark_sum("Severity_Weight").alias("Severity_Score")
    ) \
    .orderBy(col("Severity_Score").desc())

severity_by_roadtype.show(truncate=False)

+------------------+--------------+--------------+
|Road_Type         |Accident_Count|Severity_Score|
+------------------+--------------+--------------+
|Single carriageway|148543        |196689        |
|Dual carriageway  |29004         |38022         |
|Roundabout        |13196         |15648         |
|One way street    |3937          |5069          |
|Slip road         |1951          |2297          |
|Unknown           |1013          |1233          |
+------------------+--------------+--------------+



### severity score by time period

In [13]:
severity_by_timeperiod = df_severity.groupBy("Time_Period") \
    .agg(
        spark_count("Accident_Index").alias("Accident_Count"),
        spark_sum("Severity_Weight").alias("Severity_Score")
    ) \
    .orderBy(col("Severity_Score").desc())

severity_by_timeperiod.show(truncate=False)

+-----------+--------------+--------------+
|Time_Period|Accident_Count|Severity_Score|
+-----------+--------------+--------------+
|Afternoon  |66856         |86082         |
|Morning    |56379         |72461         |
|Evening    |48896         |63872         |
|Night      |15799         |21797         |
|Late Night |9714          |14746         |
+-----------+--------------+--------------+



### count ≠ severity risk

In [14]:
# Compare rank by raw count vs rank by severity score for locations
severity_by_location.withColumn(
    "Avg_Severity_Per_Accident", col("Severity_Score") / col("Accident_Count")
).orderBy(col("Avg_Severity_Per_Accident").desc()).show(15, truncate=False)

+--------------------------+--------------+--------------+-------------------------+
|Local_Authority_(District)|Accident_Count|Severity_Score|Avg_Severity_Per_Accident|
+--------------------------+--------------+--------------+-------------------------+
|Dundee City               |2             |4             |2.0                      |
|South Staffordshire       |6             |10            |1.6666666666666667       |
|South Northamptonshire    |413           |681           |1.6489104116222761       |
|Oswestry                  |30            |48            |1.6                      |
|Maldon                    |245           |389           |1.5877551020408163       |
|Daventry                  |395           |621           |1.5721518987341772       |
|Purbeck                   |265           |413           |1.558490566037736        |
|South Larkshire           |338           |524           |1.5502958579881656       |
|East Northamptonshire     |275           |425           |1.54545

## Task 5 — add these cells to the same notebook

### combination severity ranking

In [15]:
combo_severity = df_severity.groupBy(
    "Road_Type", "Speed_limit", "Weather_Conditions", "Time_Period", "Vehicle_Type"
).agg(
    spark_count("Accident_Index").alias("Accident_Count"),
    spark_sum("Severity_Weight").alias("Severity_Score")
).orderBy(col("Severity_Score").desc())

combo_severity.show(10, truncate=False)

+------------------+-----------+---------------------+-----------+------------+--------------+--------------+
|Road_Type         |Speed_limit|Weather_Conditions   |Time_Period|Vehicle_Type|Accident_Count|Severity_Score|
+------------------+-----------+---------------------+-----------+------------+--------------+--------------+
|Single carriageway|30         |Fine no high winds   |Afternoon  |Car         |25659         |32487         |
|Single carriageway|30         |Fine no high winds   |Morning    |Car         |19412         |24610         |
|Single carriageway|30         |Fine no high winds   |Evening    |Car         |17807         |22993         |
|Single carriageway|30         |Fine no high winds   |Night      |Car         |5345          |7295          |
|Single carriageway|60         |Fine no high winds   |Afternoon  |Car         |4346          |6808          |
|Single carriageway|60         |Fine no high winds   |Morning    |Car         |4164          |5980          |
|Single ca

### filter out low-sample-size noise

In [29]:
combo_severity_filtered = combo_severity.filter(col("Accident_Count") >= 30) \
    .withColumn("Avg_Severity", col("Severity_Score") / col("Accident_Count")) \
    .orderBy(col("Avg_Severity").desc())

combo_severity_filtered.show(10, truncate=False)

+------------------+-----------+---------------------+-----------+-----------------------------------+--------------+--------------+------------------+
|Road_Type         |Speed_limit|Weather_Conditions   |Time_Period|Vehicle_Type                       |Accident_Count|Severity_Score|Avg_Severity      |
+------------------+-----------+---------------------+-----------+-----------------------------------+--------------+--------------+------------------+
|Dual carriageway  |70         |Fine no high winds   |Late Night |Van / Goods 3.5 tonnes mgw or under|33            |67            |2.0303030303030303|
|Single carriageway|60         |Fine no high winds   |Late Night |Van / Goods 3.5 tonnes mgw or under|48            |92            |1.9166666666666667|
|Single carriageway|40         |Fine no high winds   |Late Night |Car                                |230           |416           |1.808695652173913 |
|Single carriageway|60         |Fine no high winds   |Evening    |Motorcycle 125cc and u

## Task 6

### build the risk measure per location within each category

In [17]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, rank, row_number

location_risk = df_severity.groupBy("Urban_or_Rural_Area", "Local_Authority_(District)") \
    .agg(
        spark_count("Accident_Index").alias("Accident_Count"),
        spark_sum("Severity_Weight").alias("Severity_Score")
    )

location_risk.show(10, truncate=False)

+-------------------+--------------------------+--------------+--------------+
|Urban_or_Rural_Area|Local_Authority_(District)|Accident_Count|Severity_Score|
+-------------------+--------------------------+--------------+--------------+
|Rural              |North Kesteven            |582           |876           |
|Urban              |East Dorset               |166           |230           |
|Rural              |South Ribble              |391           |525           |
|Rural              |Trafford                  |93            |125           |
|Urban              |Derbyshire Dales          |32            |38            |
|Urban              |Chesterfield              |474           |590           |
|Urban              |East Northamptonshire     |59            |79            |
|Rural              |West Wiltshire            |27            |31            |
|Rural              |Denbighshire              |174           |238           |
|Urban              |Kensington and Chelsea    |1376

### apply the Window function

In [18]:
window_spec = Window.partitionBy("Urban_or_Rural_Area").orderBy(col("Severity_Score").desc())

location_ranked = location_risk.withColumn(
    "Risk_Rank", dense_rank().over(window_spec)
)

top3_per_category = location_ranked.filter(col("Risk_Rank") <= 3) \
    .orderBy("Urban_or_Rural_Area", "Risk_Rank")

top3_per_category.show(20, truncate=False)

+-------------------+--------------------------+--------------+--------------+---------+
|Urban_or_Rural_Area|Local_Authority_(District)|Accident_Count|Severity_Score|Risk_Rank|
+-------------------+--------------------------+--------------+--------------+---------+
|Rural              |Cornwall                  |1992          |2584          |1        |
|Rural              |County Durham             |1421          |1903          |2        |
|Rural              |Wiltshire                 |1158          |1816          |3        |
|Urban              |Birmingham                |6099          |7717          |1        |
|Urban              |Westminster               |2811          |4341          |2        |
|Urban              |Manchester                |3072          |3772          |3        |
+-------------------+--------------------------+--------------+--------------+---------+



### partitioned by Police_Force, to show a second geographical grouping

In [19]:
location_risk_pf = df_severity.groupBy("Police_Force", "Local_Authority_(District)") \
    .agg(
        spark_count("Accident_Index").alias("Accident_Count"),
        spark_sum("Severity_Weight").alias("Severity_Score")
    )

window_spec_pf = Window.partitionBy("Police_Force").orderBy(col("Severity_Score").desc())

location_ranked_pf = location_risk_pf.withColumn(
    "Risk_Rank", dense_rank().over(window_spec_pf)
)

top3_per_pf = location_ranked_pf.filter(col("Risk_Rank") <= 3) \
    .orderBy("Police_Force", "Risk_Rank")

top3_per_pf.show(30, truncate=False)

+------------------+--------------------------+--------------+--------------+---------+
|Police_Force      |Local_Authority_(District)|Accident_Count|Severity_Score|Risk_Rank|
+------------------+--------------------------+--------------+--------------+---------+
|Bedfordshire      |Central Bedfordshire      |1225          |1691          |1        |
|Bedfordshire      |Luton                     |998           |1224          |2        |
|Bedfordshire      |Bedford                   |792           |1016          |3        |
|City of London    |City of London            |635           |977           |1        |
|Cleveland         |Stockton-on-Tees          |628           |858           |1        |
|Cleveland         |Middlesbrough             |597           |737           |2        |
|Cleveland         |Redcar and Cleveland      |444           |610           |3        |
|Cumbria           |Eden                      |1             |1             |1        |
|Derbyshire        |Derby       

## Task 7

### build the raw components per location

In [20]:
from pyspark.sql.functions import avg as spark_avg

# Component 1 & 2: frequency and severity (reuse from Task 6, but at national level this time)
location_components = df_severity.groupBy("Local_Authority_(District)").agg(
    spark_count("Accident_Index").alias("Accident_Count"),
    spark_sum("Severity_Weight").alias("Severity_Score")
)

# Component 3: dangerous conditions rate — % of accidents in non-"Fine"/non-"Dry" conditions
df_severity = df_severity.withColumn(
    "Is_Dangerous_Condition",
    when(
        (col("Weather_Conditions") != "Fine no high winds") |
        (col("Road_Surface_Conditions") != "Dry"),
        1
    ).otherwise(0)
)

danger_rate = df_severity.groupBy("Local_Authority_(District)").agg(
    spark_avg("Is_Dangerous_Condition").alias("Dangerous_Condition_Rate")
)

location_components = location_components.join(danger_rate, on="Local_Authority_(District)", how="left")
location_components.show(10, truncate=False)

+--------------------------+--------------+--------------+------------------------+
|Local_Authority_(District)|Accident_Count|Severity_Score|Dangerous_Condition_Rate|
+--------------------------+--------------+--------------+------------------------+
|Worcester                 |433           |489           |0.3117782909930716      |
|North Wiltshire           |70            |96            |0.6142857142857143      |
|North Kesteven            |705           |1035          |0.49361702127659574     |
|Waveney                   |593           |811           |0.3524451939291737      |
|Epping Forest             |924           |1312          |0.36363636363636365     |
|North Cornwall            |58            |74            |0.5689655172413793      |
|Maldon                    |245           |389           |0.40816326530612246     |
|St. Edmundsbury           |562           |778           |0.43416370106761565     |
|Guildford                 |1182          |1486          |0.4103214890016921

### normalize each component to 0–1

In [21]:
from pyspark.sql.functions import min as spark_min, max as spark_max, lit

def minmax_normalize(df, colname, new_colname):
    stats = df.agg(spark_min(colname).alias("min_val"), spark_max(colname).alias("max_val")).collect()[0]
    min_val, max_val = stats["min_val"], stats["max_val"]
    return df.withColumn(
        new_colname,
        (col(colname) - lit(min_val)) / (lit(max_val) - lit(min_val))
    )

location_components = minmax_normalize(location_components, "Accident_Count", "Norm_Frequency")
location_components = minmax_normalize(location_components, "Severity_Score", "Norm_Severity")
location_components = minmax_normalize(location_components, "Dangerous_Condition_Rate", "Norm_Danger")

location_components.select(
    "Local_Authority_(District)", "Norm_Frequency", "Norm_Severity", "Norm_Danger"
).show(10, truncate=False)

+--------------------------+--------------------+--------------------+-------------------+
|Local_Authority_(District)|Norm_Frequency      |Norm_Severity       |Norm_Danger        |
+--------------------------+--------------------+--------------------+-------------------+
|Worcester                 |0.07008436080467229 |0.06253203485392107 |0.3117782909930716 |
|North Wiltshire           |0.011194029850746268|0.012173244490005125|0.6142857142857143 |
|North Kesteven            |0.11421155094094744 |0.13249615581752947 |0.49361702127659574|
|Waveney                   |0.09604153147306943 |0.10379292670425423 |0.3524451939291737 |
|Epping Forest             |0.14974042829331602 |0.16799077396207074 |0.36363636363636365|
|North Cornwall            |0.009247242050616482|0.009354177344951307|0.5689655172413793 |
|Maldon                    |0.03958468526930565 |0.04971809328549462 |0.40816326530612246|
|St. Edmundsbury           |0.0910123296560675  |0.0995643259866735  |0.43416370106761565|

### combine into a weighted Risk Score

In [30]:
location_components = location_components.withColumn(
    "Road_Safety_Risk_Score",
    (col("Norm_Severity") * 0.5) +
    (col("Norm_Frequency") * 0.35) +
    (col("Norm_Danger") * 0.15)
)

ranked_risk = location_components.orderBy(col("Road_Safety_Risk_Score").desc())
ranked_risk.select(
    "Local_Authority_(District)", "Accident_Count", "Severity_Score",
    "Dangerous_Condition_Rate", "Road_Safety_Risk_Score"
).show(15, truncate=False)

+--------------------------+--------------+--------------+------------------------+----------------------+
|Local_Authority_(District)|Accident_Count|Severity_Score|Dangerous_Condition_Rate|Road_Safety_Risk_Score|
+--------------------------+--------------+--------------+------------------------+----------------------+
|Birmingham                |6165          |7805          |0.36253041362530414     |0.9043795620437955    |
|Manchester                |3132          |3854          |0.39655172413793105     |0.484125627167444     |
|Westminster               |2811          |4341          |0.2113127001067236      |0.46931492050316614   |
|Sheffield                 |2750          |3464          |0.3618181818181818      |0.43223794902127066   |
|Liverpool                 |2611          |3445          |0.32707774798927614     |0.4179169572915748    |
|Cornwall                  |2606          |3288          |0.3883346124328473      |0.41676263627271015   |
|County Durham             |2228     

## Task 8

### explain a groupBy+agg (triggers a shuffle)

In [23]:
df_severity.groupBy("Local_Authority_(District)") \
    .agg(
        spark_count("Accident_Index").alias("Accident_Count"),
        spark_sum("Severity_Weight").alias("Severity_Score")
    ).explain(True)

== Parsed Logical Plan ==
'Aggregate ['Local_Authority_(District)], ['Local_Authority_(District), 'count('Accident_Index) AS Accident_Count#3616, 'sum('Severity_Weight) AS Severity_Score#3617]
+- Project [Accident_Index#17, Accident Date#18, Month#19, Day_of_Week#20, Year#21, Junction_Control#22, Junction_Detail#23, Accident_Severity#24, Latitude#25, Light_Conditions#26, Local_Authority_(District)#27, Carriageway_Hazards#28, Longitude#29, Number_of_Casualties#30, Number_of_Vehicles#31, Police_Force#32, Road_Surface_Conditions#44, Road_Type#45, Speed_limit#35, Time#41, Urban_or_Rural_Area#37, Weather_Conditions#46, Vehicle_Type#39, Accident_Date_Parsed#42, Time_of_Day#43, ... 6 more fields]
   +- Project [Accident_Index#17, Accident Date#18, Month#19, Day_of_Week#20, Year#21, Junction_Control#22, Junction_Detail#23, Accident_Severity#24, Latitude#25, Light_Conditions#26, Local_Authority_(District)#27, Carriageway_Hazards#28, Longitude#29, Number_of_Casualties#30, Number_of_Vehicles#31, 

### explain the Window function

In [24]:
location_ranked.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(Risk_Rank, 'dense_rank() windowspecdefinition('Urban_or_Rural_Area, 'Severity_Score DESC NULLS LAST, unspecifiedframe$()), None)]
+- Aggregate [Urban_or_Rural_Area#37, Local_Authority_(District)#27], [Urban_or_Rural_Area#37, Local_Authority_(District)#27, count(Accident_Index#17) AS Accident_Count#1940L, sum(Severity_Weight#865) AS Severity_Score#1941L]
   +- Project [Accident_Index#17, Accident Date#18, Month#19, Day_of_Week#20, Year#21, Junction_Control#22, Junction_Detail#23, Accident_Severity#24, Latitude#25, Light_Conditions#26, Local_Authority_(District)#27, Carriageway_Hazards#28, Longitude#29, Number_of_Casualties#30, Number_of_Vehicles#31, Police_Force#32, Road_Surface_Conditions#44, Road_Type#45, Speed_limit#35, Time#41, Urban_or_Rural_Area#37, Weather_Conditions#46, Vehicle_Type#39, Accident_Date_Parsed#42, Time_of_Day#43, ... 5 more fields]
      +- Project [Accident_Index#17, Accident Date#18, Month#19, Day_of_W

### explain a filter + select

In [25]:
df_clean.filter(col("Speed_limit") > 30).select("Accident_Index", "Speed_limit").explain(True)

== Parsed Logical Plan ==
'Project ['Accident_Index, 'Speed_limit]
+- Filter (Speed_limit#35 > 30)
   +- Project [Accident_Index#17, Accident Date#18, Month#19, Day_of_Week#20, Year#21, Junction_Control#22, Junction_Detail#23, Accident_Severity#24, Latitude#25, Light_Conditions#26, Local_Authority_(District)#27, Carriageway_Hazards#28, Longitude#29, Number_of_Casualties#30, Number_of_Vehicles#31, Police_Force#32, Road_Surface_Conditions#44, Road_Type#45, Speed_limit#35, Time#41, Urban_or_Rural_Area#37, CASE WHEN (isnull(Weather_Conditions#38) OR (Weather_Conditions#38 = )) THEN Unknown ELSE Weather_Conditions#38 END AS Weather_Conditions#46, Vehicle_Type#39, Accident_Date_Parsed#42, Time_of_Day#43]
      +- Project [Accident_Index#17, Accident Date#18, Month#19, Day_of_Week#20, Year#21, Junction_Control#22, Junction_Detail#23, Accident_Severity#24, Latitude#25, Light_Conditions#26, Local_Authority_(District)#27, Carriageway_Hazards#28, Longitude#29, Number_of_Casualties#30, Number_of_V

### demonstrate caching, and time the difference

In [26]:
import time

# Without cache — df_severity gets recomputed from scratch every action
start = time.time()
df_severity.groupBy("Time_Period").count().collect()
df_severity.groupBy("Road_Type").count().collect()
print("Without cache:", time.time() - start, "seconds")

# With cache
df_severity_cached = df_severity.cache()
df_severity_cached.count()  # materializes the cache

start = time.time()
df_severity_cached.groupBy("Time_Period").count().collect()
df_severity_cached.groupBy("Road_Type").count().collect()
print("With cache:", time.time() - start, "seconds")

Without cache: 4.75312352180481 seconds
With cache: 6.092169761657715 seconds


## Link to Spark Context Web

In [27]:
print(spark.sparkContext.uiWebUrl)

http://rwibutso:4040
